In [1]:
import re
from collections import Counter

import pandas as pd

# Load data
df = pd.read_csv("imdb_processed.csv")


In [2]:
df["Description"] = df["Description"].fillna("")
df["Movie Name"] = df["Movie Name"].fillna("")
df["Movie Name Clean"] = df["Movie Name"].str.replace(r"^\s*\d+\.\s*", "", regex=True)


In [3]:
df.head()

,Movie Name,Duration,IMdb rating,Rating,Description,Votes,Poster,Month of Release,Movie Name Clean
0,1. Vindhya Victim Verdict V3,2h 7m,7.6,595,V3 is a fight for their rights and against an ...,595,https://m.media-amazon.com/images/M/MV5BYTE5Yj...,January,Vindhya Victim Verdict V3
1,2. Thunivu,2h 26m,6.1,29K,A mysterious mastermind - Daredevil and his te...,"28,737",https://m.media-amazon.com/images/M/MV5BY2M0OD...,January,Thunivu
2,3. Varisu,2h 49m,6.0,37K,Vijay Rajendran is a happy to-go lucky man. Th...,"36,755",https://m.media-amazon.com/images/M/MV5BMjE4Yj...,January,Varisu
3,4. Oangaram,1h 52m,NaN,NaN,,NaN,NaN,January,Oangaram
4,5. Vallavanukkum Vallavan,2h 47m,7.5,13,"Two conmen, who believe that money can solve a...",13,https://m.media-amazon.com/images/M/MV5BZGRkMG...,January,Vallavanukkum Vallavan


In [4]:
stopwords = {
    "the",
    "a",
    "an",
    "and",
    "or",
    "of",
    "to",
    "in",
    "on",
    "for",
    "with",
    "by",
    "from",
    "their",
    "his",
    "her",
    "its",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "as",
    "at",
    "that",
    "this",
    "these",
    "those",
    "it",
    "they",
    "them",
    "he",
    "she",
    "you",
    "your",
    "our",
    "us",
    "after",
    "before",
    "when",
    "who",
    "whom",
    "which",
    "while",
    "into",
    "than",
    "then",
    "there",
    "here",
    "about",
    "against",
    "over",
    "under",
    "through",
    "during",
    "within",
    "without",
    "between",
    "two",
    "one",
}

In [5]:
def tokenize(text):
    words = re.findall(r"[a-zA-Z0-9]+", str(text).lower())
    return [w for w in words if w not in stopwords]

In [6]:
doc_tokens = df["Description"].map(tokenize)

In [7]:
doc_tokens

0      [v3, fight, rights, injustice, gang, rape, mur...
1      [mysterious, mastermind, daredevil, team, form...
2      [vijay, rajendran, happy, go, lucky, man, thin...
3                                                     []
4      [conmen, believe, money, can, solve, all, prob...
                             ...                        
204    [wife, sends, him, rehab, alcohol, addiction, ...
205    [1975, filmmaker, agrees, collaborate, film, g...
206    [daring, robbery, jewellery, showroom, kicksta...
207                                                   []
208    [kathir, s, wife, chaitra, mentally, affected,...
Name: Description, Length: 209, dtype: object

In [8]:
doc_lengths = doc_tokens.map(len).replace(0, 1)

In [9]:
doc_lengths

0      22
1      15
2      16
3       1
4      17
       ..
204    22
205    10
206    12
207     1
208    27
Name: Description, Length: 209, dtype: int64

In [10]:
for word, length in zip(doc_tokens, doc_lengths):
    print(word, length)
    

['v3', 'fight', 'rights', 'injustice', 'gang', 'rape', 'murder', 'encountered', 'case', 'achieving', 'poetic', 'justice', 'help', 'system', 'fails', 'initially', 'due', 'undue', 'pressure', 'representatives', 'thy', 'people'] 22
['mysterious', 'mastermind', 'daredevil', 'team', 'forms', 'plan', 'commits', 'bank', 'heist', 'find', 'corporate', 'looted', 'people', 's', 'money'] 15
['vijay', 'rajendran', 'happy', 'go', 'lucky', 'man', 'things', 'change', 'father', 'becomes', 'terminally', 'ill', 'left', 'manage', 'business', 'empire'] 16
[] 1
['conmen', 'believe', 'money', 'can', 'solve', 'all', 'problems', 'decide', 'extract', 'money', 'wrongdoers', 'trouble', 'begins', 'mad', 'policeman', 'crosses', 'way'] 17
['tale', 'young', 'man', 'journey', 'love', 'life', 'understand', 'meaning', 'eternal', 'love', 'purpose', 'emotional', 'roller', 'coaster', 'presented', 'touch', 'fantasy'] 17
['people', 'autistic', 'teenager', 'living', 'mother', 'young', 'woman', 'abducted', 'unknown', 'group'] 

In [11]:
tf_docs = []
for tokens, length in zip(doc_tokens, doc_lengths):
    counts = Counter(tokens)
    tf = {term: count / length for term, count in counts.items()}
    tf_docs.append(tf)

In [12]:
tf_docs[1]

{'mysterious': 0.06666666666666667,
 'mastermind': 0.06666666666666667,
 'daredevil': 0.06666666666666667,
 'team': 0.06666666666666667,
 'forms': 0.06666666666666667,
 'plan': 0.06666666666666667,
 'commits': 0.06666666666666667,
 'bank': 0.06666666666666667,
 'heist': 0.06666666666666667,
 'find': 0.06666666666666667,
 'corporate': 0.06666666666666667,
 'looted': 0.06666666666666667,
 'people': 0.06666666666666667,
 's': 0.06666666666666667,
 'money': 0.06666666666666667}

In [19]:
def search_movies(query, top_k=5):
    q_tokens = tokenize(query)
    if not q_tokens:
        return []

    q_counts = Counter(q_tokens)
    q_len = len(q_tokens)
    q_tf = {term: count / q_len for term, count in q_counts.items()}
    # print(query, q_tf)
    
    scores = []
    for idx, doc_tf in enumerate(tf_docs):
        score = 0.0
        # print(doc_tf)
        for term, q_weight in q_tf.items():
            score += q_weight * doc_tf.get(term, 0.0)
        # print(score)
        # input("Enter to proceed")
        if score > 0:
            scores.append((score, idx))
    

    scores.sort(reverse=True)
    results = []
    for score, idx in scores[:top_k]:
        results.append(
            {
                "movie": df.loc[idx, "Movie Name Clean"],
                "score": round(score, 4),
                "description": df.loc[idx, "Description"],
            }
        )
    return results

In [21]:
queries = [
    "bank heist money",
    "murder investigation",
    "love family",
    "alien invasion",
    "crime gangster",
]

for q in queries:
    print(f"\nQUERY: {q}")
    for row in search_movies(q, top_k=10):
        print( row["movie"])




QUERY: bank heist money
Thunivu
Kasethan Kadavulada
Vallavanukkum Vallavan
Takkar
Pichaikkaran 2
Paramporul
DD Returns
Vasantha Mullai
Screenplay
Pudhu Vedham

QUERY: murder investigation
Run Baby Run
Kidugu
Oongi Adicha Ondra Ton Weightu Da
Memories
Maruthi Nagar Police Station
Ra Ra Sarasukku Ra Ra
Raakadhan
Paramporul
Kuttrappinnani
Narkaram

QUERY: love family
Irumban
Kuttram Purindhaal
Karumegangal Kalaiginrana
Adai Mazhai Kaalam
Kanneera
Erumbu
Munthirikkaadu
Rangoli
1982 Anbarasin Kaadhal
Love

QUERY: alien invasion
Ellaam Mela Irukuravan Paathuppan

QUERY: crime gangster
Shoot the Kuruvi
Pathu Thala
Jigarthanda Double X
Agilan
Dinosaurs
Kannai Nambathey
Wanted: Shades of Life ft. Genkeswaran Muniyan
Infinity
Thalainagaram 2
Kather Basha Endra Muthuramalingam
